### GROUP BY vs Windows Functions

"GROUP BY and Window Functions are both used to perform calculations on data, but the main difference is how they handle the result set.

GROUP BY combines multiple rows into a single row for each group. We generally use it when we need aggregated results, such as department-wise average salary, total sales, or employee count.

Window Functions, on the other hand, perform calculations across a set of related rows without reducing the number of rows in the result set. We use them when we need to calculate rankings, running totals, department-wise averages, or compare current and previous records.

For example, if I want to calculate the average salary department-wise and display only one record per department, I use GROUP BY.

But if I want to display every employee along with their department's average salary, I use a Window Function with PARTITION BY.

So, the key difference is that GROUP BY reduces rows, whereas Window Functions retain individual rows while performing calculations."

In [0]:
%sql
--row_number(),rank(),dense_rank(),lag(),lead()
select * from (
select e.*,
row_number() over(partition by e.department order by e.salary desc) as rn 
from interview.`25_sql`.employees e 
) t
where rn=1;

In [0]:
%sql
--Fetch the first 2 employees from each department to join the company.
SELECT * FROM (
    SELECT e.*,row_number() over(partition by e.department order by e.empid) as rn
    FROM interview.`25_sql`.employees e
) as t
WHERE t.rn<3;

In [0]:
%sql
--Fetch the top 3 employees in each department the max salary
SELECT * FROM
    (SELECT *, rank() over(partition by department order by salary desc) as rnk FROM interview.`25_sql`.employees) AS x
WHERE x.rnk<=3;

In [0]:
%sql
--dense_rank
SELECT * FROM
(SELECT empname,salary,dense_rank() over(partition by department order by salary desc ) as dense_rank FROM interview.`25_sql`.employees) as t
WHERE dense_rank=1

## LAG and LEAD

In [0]:
%sql
--Fetch a query to display if the salary of an employee is higher, lower or equal to the previous employee.
--lag syntax LAG(col_nm,set_value,default) set_value means previous rows
SELECT e.*,LAG(e.salary) over(partition by e.department order by e.empid) as prev_sal,
LEAD(e.salary) over(partition by e.department order by e.empid) as next_sal 
FROM interview.`25_sql`.employees e;


#### first_value()

In [0]:
%sql
-- Display the most expensive product under each category (corresponding to each record)
-- first_value() must be given a column (the value to carry from the first row of each partition)
SELECT i.*,
       first_value(i.product_id) OVER (
           PARTITION BY i.product_category 
           ORDER BY i.product_price DESC
       ) AS most_expensive_product
FROM interview.`25_sql`.inventory i;

In [0]:
%sql
-- last_value()
-- Display the least expensive product under each category (corresponding to each record)
-- last_value() needs the frame ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING,
-- otherwise it only looks at rows up to the current row (default frame)
SELECT i.*,
       last_value(i.product_id) OVER (
           PARTITION BY i.product_category 
           ORDER BY i.product_price ASC
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS least_expensive_product
FROM interview.`25_sql`.inventory i;